# Training Transformer Model for KeyWord Spot Task
Notebook này được sử dụng để huấn luyện mô hình Transformer phù hợp cho task keyword spot của ĐAĐN (CO3107).
Thành viên của nhóm bao gồm:
+ Lại Nguyễn Hoàng Hưng - 2311327
+ Đoàn Võ Việt Khôi - 2311660
+ Huỳnh Đức Nhân - 2312420
+ Phạm Trần Minh Trí - 2313622

## 1. Download Data
Dữ liệu được sử dụng để train model là [Google Speech Commands](https://www.kaggle.com/datasets/neehakurelli/google-speech-commands) từ Kaggle.

Tuy nhiên, vì mục tiêu chỉ là 4 từ Up, Down, Left, Right, nhóm đã tiến hành điều chỉnh dataset gốc để phù hợp với mục tiêu hơn. Cấu trúc dataset bây giờ như sau:
+ Tổng số mẫu: 11802 mẫu, 5 nhãn.
+ Nhãn Down: 2359 mẫu.
+ Nhãn Left: 2353 mẫu.
+ Nhãn Other: 2348 mẫu.
+ Nhãn Right: 2367 mẫu.
+ Nhãn Up: 2375 mẫu.

Trong đó, 4 nhãn mục tiêu (Down, Left, Right, Up) được lấy từ dataset gốc; nhãn Other được lấy ngẫu nhiên từ 20 từ còn lại trong dataset gốc, kèm theo các loại nhiễu.

In [ ]:
import os
import zipfile
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
import gdown

zip_file_path = './keyword_data.zip'
extract_path = './keyword_spot/'
data_path = './keyword_spot/keyword_data/'

if not os.path.exists(zip_file_path):
  !gdown 1ggzXjSqezerQepd9775cdyptsIE-aSB4

with zipfile.ZipFile(zip_file_path, 'r') as zip_ref:
    zip_ref.extractall(extract_path)

print(f'Data extracted to: {extract_path}')

Downloading...
From (original): https://drive.google.com/uc?id=1ggzXjSqezerQepd9775cdyptsIE-aSB4
From (redirected): https://drive.google.com/uc?id=1ggzXjSqezerQepd9775cdyptsIE-aSB4&confirm=t&uuid=253dc24b-2393-4aea-af92-9bcd1305b882
To: /content/keyword_data.zip
100% 265M/265M [00:02<00:00, 125MB/s]
Data extracted to: ./keyword_spot/


In [ ]:
import random
np.random.seed(42)
random.seed(42)
import tensorflow as tf
tf.random.set_seed(42)

NUM_CLASSES = 5
TRAIN = False
TUNE_FE = True
TUNE_MODEL = False

In [ ]:
# Lấy data sau khi unzip
audio_data = []
labels = []

for label_dir in os.listdir(data_path):
    label_path = os.path.join(data_path, label_dir)
    if os.path.isdir(label_path):
        for audio_file in os.listdir(label_path):
            if audio_file.endswith('.wav'):
                audio_data.append(os.path.join(label_path, audio_file))
                labels.append(label_dir)

# Tạo dataframe chứa đường dẫn và nhãn của các mẫu
df = pd.DataFrame({
    'filepath': audio_data,
    'label': labels
})

print(f'Total audio files: {len(df)}')
display(df.head())

Total audio files: 11802


,filepath,label
0,./keyword_spot/keyword_data/up/7dc95912_nohash...,up
1,./keyword_spot/keyword_data/up/28ce0c58_nohash...,up
2,./keyword_spot/keyword_data/up/333784b7_nohash...,up
3,./keyword_spot/keyword_data/up/91cdef62_nohash...,up
4,./keyword_spot/keyword_data/up/21832144_nohash...,up


In [ ]:
# Encode labels
le = LabelEncoder()
df['encoded_label'] = le.fit_transform(df['label'])

# Split data into training and testing sets
X = df['filepath']
y = df['encoded_label']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

print(f'Train set size: {len(X_train)}')
print(f'Test set size: {len(X_test)}')
print(f'Labels: {le.classes_}')

Train set size: 9441
Test set size: 2361
Labels: ['down' 'left' 'other' 'right' 'up']


## 2. Feature Extraction

Tính toán sơ bộ Global Mean và Global Std của tập train thông qua các bước:
+ Pre-emphasis: Cân bằng năng lượng phổ của âm thanh
+ Framing: Chia nhỏ tín hiệu âm thanh thành các khung 25ms với bước nhảy 10ms
+ Windowing: Làm mịn biên độ 2 đầu khung tín hiệu về mức 0.
+ Biến đổi Fourier: Chuyển đổi dữ liệu từ miền thời gian sang miền tần số.
+ Mel Filter Bank: Phổ năng lượng được đưa qua bộ lọc nhằm mô phỏng biên độ nghe của tai người.

Sau khi đã có Global mean và Global Std, thực hiện lại pipeline trích xuất tương tự, nhưng thêm chuẩn hóa Z-score để triệt tiêu khuếch đại nhiễu.

In [ ]:
import numpy as np
import scipy.io.wavfile as wav
from scipy.fftpack import dct
import os

# Configuration class for feature extraction
class FEConfig:
    def __init__(self, nfilt=40, nfft=512, target_time_steps=100):
        self.nfilt = nfilt                 # Number of mel filter banks
        self.nfft = nfft                   # FFT size
        self.target_time_steps = target_time_steps  # Padded time dimension

    def __repr__(self):
        return f"FEConfig(nfilt={self.nfilt}, nfft={self.nfft}, target_time_steps={self.target_time_steps})"

def extract_mel_spectrogram_raw(file_path, fe_config=None):
    """Extract raw mel spectrogram without normalization (for computing mean/std)"""
    if fe_config is None:
        fe_config = FEConfig()

    if not os.path.exists(file_path):
        raise FileNotFoundError(f"Không tìm thấy file: {file_path}")

    sample_rate, signal = wav.read(file_path)
    if signal.dtype == np.int16:
        signal = signal.astype(np.float32)

    pre_emphasis = 0.97
    emphasized_signal = np.append(signal[0], signal[1:] - pre_emphasis * signal[:-1])

    frame_size, frame_stride = 0.025, 0.01
    frame_length, frame_step = frame_size * sample_rate, frame_stride * sample_rate
    signal_length = len(emphasized_signal)
    frame_length = int(round(frame_length))
    frame_step = int(round(frame_step))
    num_frames = int(np.ceil(float(np.abs(signal_length - frame_length)) / frame_step))

    pad_signal_length = num_frames * frame_step + frame_length
    z = np.zeros((pad_signal_length - signal_length))
    pad_signal = np.append(emphasized_signal, z)

    indices = np.tile(np.arange(0, frame_length), (num_frames, 1)) + \
              np.tile(np.arange(0, num_frames * frame_step, frame_step), (frame_length, 1)).T
    frames = pad_signal[indices.astype(np.int32, copy=False)]
    frames *= np.hamming(frame_length)

    mag_frames = np.absolute(np.fft.rfft(frames, fe_config.nfft))
    pow_frames = ((1.0 / fe_config.nfft) * (mag_frames ** 2))

    low_freq_mel = 0
    high_freq_mel = (2595 * np.log10(1 + (sample_rate / 2) / 700))
    mel_points = np.linspace(low_freq_mel, high_freq_mel, fe_config.nfilt + 2)
    hz_points = (700 * (10**(mel_points / 2595) - 1))
    bin = np.floor((fe_config.nfft + 1) * hz_points / sample_rate)

    fbank = np.zeros((fe_config.nfilt, int(np.floor(fe_config.nfft / 2 + 1))))
    for m in range(1, fe_config.nfilt + 1):
        f_m_minus = int(bin[m - 1])
        f_m = int(bin[m])
        f_m_plus = int(bin[m + 1])
        for k in range(f_m_minus, f_m):
            fbank[m - 1, k] = (k - bin[m - 1]) / (bin[m] - bin[m - 1])
        for k in range(f_m, f_m_plus):
            fbank[m - 1, k] = (bin[m + 1] - k) / (bin[m + 1] - bin[m])

    mel_spectrogram = np.dot(pow_frames, fbank.T)
    mel_spectrogram = np.where(mel_spectrogram == 0, np.finfo(float).eps, mel_spectrogram)
    mel_spectrogram = 20 * np.log10(mel_spectrogram)

    current_frames = mel_spectrogram.shape[0]
    if current_frames < fe_config.target_time_steps:
        pad_width = ((0, fe_config.target_time_steps - current_frames), (0, 0))
        mel_padded = np.pad(mel_spectrogram, pad_width, mode='constant', constant_values=0)
    else:
        indices = np.linspace(0, current_frames - 1, fe_config.target_time_steps).astype(int)
        mel_padded = mel_spectrogram[indices]

    return mel_padded  # Shape: (target_time_steps, nfilt)

In [ ]:
def extract_mel_spectrogram(file_path, fe_config=None, global_mean=None, global_std=None):
    """Extract mel spectrogram with normalization using provided global_mean and global_std"""
    if fe_config is None:
        fe_config = FEConfig()

    if not os.path.exists(file_path):
        raise FileNotFoundError(f"Không tìm thấy file: {file_path}")

    sample_rate, signal = wav.read(file_path)

    if signal.dtype == np.int16:
        signal = signal.astype(np.float32)

    pre_emphasis = 0.97
    emphasized_signal = np.append(signal[0], signal[1:] - pre_emphasis * signal[:-1])

    # Framing
    frame_size, frame_stride = 0.025, 0.01
    frame_length, frame_step = frame_size * sample_rate, frame_stride * sample_rate
    signal_length = len(emphasized_signal)
    frame_length = int(round(frame_length))
    frame_step = int(round(frame_step))
    num_frames = int(np.ceil(float(np.abs(signal_length - frame_length)) / frame_step))

    pad_signal_length = num_frames * frame_step + frame_length
    z = np.zeros((pad_signal_length - signal_length))
    pad_signal = np.append(emphasized_signal, z)

    indices = np.tile(np.arange(0, frame_length), (num_frames, 1)) + \
              np.tile(np.arange(0, num_frames * frame_step, frame_step), (frame_length, 1)).T
    frames = pad_signal[indices.astype(np.int32, copy=False)]

    # Windowing (Hamming)
    frames *= np.hamming(frame_length)

    # FFT and Power Spectrum
    mag_frames = np.absolute(np.fft.rfft(frames, fe_config.nfft))
    pow_frames = ((1.0 / fe_config.nfft) * (mag_frames ** 2))

    # Filter Banks
    low_freq_mel = 0
    high_freq_mel = (2595 * np.log10(1 + (sample_rate / 2) / 700))
    mel_points = np.linspace(low_freq_mel, high_freq_mel, fe_config.nfilt + 2)
    hz_points = (700 * (10**(mel_points / 2595) - 1))
    bin = np.floor((fe_config.nfft + 1) * hz_points / sample_rate)

    fbank = np.zeros((fe_config.nfilt, int(np.floor(fe_config.nfft / 2 + 1))))
    for m in range(1, fe_config.nfilt + 1):
        f_m_minus = int(bin[m - 1])
        f_m = int(bin[m])
        f_m_plus = int(bin[m + 1])
        for k in range(f_m_minus, f_m):
            fbank[m - 1, k] = (k - bin[m - 1]) / (bin[m] - bin[m - 1])
        for k in range(f_m, f_m_plus):
            fbank[m - 1, k] = (bin[m + 1] - k) / (bin[m + 1] - bin[m])

    filter_banks = np.dot(pow_frames, fbank.T)
    filter_banks = np.where(filter_banks == 0, np.finfo(float).eps, filter_banks)
    filter_banks = 20 * np.log10(filter_banks)  # dB

    # Normalization
    if global_mean is not None and global_std is not None:
        mel_spectrogram = (filter_banks - global_mean) / (global_std + 1e-8)
    else:
        mel_spectrogram = filter_banks

    # Pad/Interpolate to target time steps
    current_frames = mel_spectrogram.shape[0]
    if current_frames < fe_config.target_time_steps:
        pad_width = ((0, fe_config.target_time_steps - current_frames), (0, 0))
        mel_padded = np.pad(mel_spectrogram, pad_width, mode='constant', constant_values=0)
    else:
        indices = np.linspace(0, current_frames - 1, fe_config.target_time_steps).astype(int)
        mel_padded = mel_spectrogram[indices]

    return mel_padded  # Shape: (target_time_steps, nfilt)

## 3. Transformer Model
Xây dựng mô hình transformer Classification cho dữ liệu 2D (spectrogram). Các đặc điểm chính bao gồm:
+ Sử dụng CNN để tạo patch.
+ Dùng Learnable Positional Encoding.
+ Sử dụng Custom Multi-head Attention được xây dựng để chạy trên ESP32.
+ Sử dụng BatchNorm thay LayerNorm vì không hỗ trợ.

In [ ]:
import tensorflow as tf
from tensorflow.keras import layers
import numpy as np

# Configuration class for model
class ModelConfig:
    def __init__(self, input_shape=(100, 40, 1), num_classes=NUM_CLASSES, embed_dim=64, num_heads=4, ff_dim=256):
        self.input_shape = input_shape
        self.num_classes = num_classes
        self.embed_dim = embed_dim
        self.num_heads = num_heads
        self.ff_dim = ff_dim

    def __repr__(self):
        return f"ModelConfig(input_shape={self.input_shape}, num_classes={self.num_classes}, embed_dim={self.embed_dim}, num_heads={self.num_heads}, ff_dim={self.ff_dim})"

# Config cho model có kích thước 110k param, cũng chính là model được sử dụng
model_config = ModelConfig(
    input_shape=(100, 40, 1),
    num_classes=NUM_CLASSES,
    embed_dim=64,
    num_heads=4,
    ff_dim=256
)

print(f"Model Config: {model_config}")

Model Config: ModelConfig(input_shape=(100, 40, 1), num_classes=5, embed_dim=64, num_heads=4, ff_dim=256)


In [ ]:
def custom_multihead_attention(x, num_heads, key_dim):
    """
    Hiện thực Multi-Head Attention tối ưu để chạy trên Vi điều khiển
    Sử dụng các lớp cơ bản (Multiply, Conv2D, Reshape) để thay thế BATCH_MATMUL (TFLite Micro trên ESP32 không hỗ trợ BATCH_MATMUL)
    """
    L = int(x.shape[1])
    embed_dim = int(x.shape[2])

    head_outputs = []

    for h in range(num_heads):
        # 1. Chiếu các vector Q, K, V cho từng Head (Dùng Dense hỗ trợ 100%)
        q = layers.Dense(key_dim)(x) # Shape: (batch, L, key_dim)
        k = layers.Dense(key_dim)(x)
        v = layers.Dense(key_dim)(x)

        # 2. Tính Q * K^T bằng thủ thuật Broadcasting thay vì BATCH_MATMUL
        q_exp = layers.Reshape((L, 1, key_dim))(q)
        k_exp = layers.Reshape((1, L, key_dim))(k)

        # Multiply phần tử -> (batch, L, L, key_dim)
        qk_mul = layers.Multiply()([q_exp, k_exp])

        # 3. Tính tổng (Sum) theo trục key_dim bằng Conv2D (1x1 conv)
        # Kết hợp chia cho sqrt(key_dim) ngay trong trọng số Kernel
        scale = 1.0 / np.sqrt(key_dim)
        sum_kernel = np.ones((1, 1, key_dim, 1), dtype=np.float32) * scale

        qk_sum = layers.Conv2D(
            filters=1,
            kernel_size=(1, 1),
            use_bias=False,
            trainable=False,
            kernel_initializer=tf.keras.initializers.Constant(sum_kernel)
        )(qk_mul) # Shape: (batch, L, L, 1)

        qk_sum = layers.Reshape((L, L))(qk_sum) # Shape: (batch, L, L)

        # 4. Tính toán Attention Score (Hỗ trợ 100%)
        attn_scores = layers.Softmax(axis=-1)(qk_sum)

        # 5. Nhân Attn_scores * V bằng Broadcasting
        attn_exp = layers.Reshape((L, L, 1))(attn_scores)
        v_exp = layers.Reshape((1, L, key_dim))(v)

        av_mul = layers.Multiply()([attn_exp, v_exp]) # Shape: (batch, L, L, key_dim)

        # 6. Tính tổng theo trục L bằng DepthwiseConv2D
        dw_kernel = np.ones((1, L, key_dim, 1), dtype=np.float32)

        av_sum = layers.DepthwiseConv2D(
            kernel_size=(1, L),
            strides=(1, 1),
            padding='valid',
            use_bias=False,
            trainable=False,
            depthwise_initializer=tf.keras.initializers.Constant(sum_kernel)
        )(av_mul) # Shape: (batch, L, 1, key_dim)

        # 7. Reshape trả lại dạng ban đầu (batch, L, key_dim)
        head_out = layers.Reshape((L, key_dim))(av_sum)
        head_outputs.append(head_out)

    # 8. Gộp các Head lại
    if num_heads > 1:
        multi_head = layers.Concatenate(axis=-1)(head_outputs)
    else:
        multi_head = head_outputs[0]

    # 9. Lớp Linear chiếu lần cuối
    out = layers.Dense(embed_dim)(multi_head)

    return out


def build_kws_transformer_light(model_config):
    """
    Xây dựng model Transofmer có kích thước theo model_config
    Sử dụng CNN để tạo patch, sử dụng BatchNorm thay Layer vì ESP32 không hỗ trợ LayerNorm
    """
    inputs = layers.Input(shape=model_config.input_shape)

    # 1. Chia patch làm token
    x = layers.Conv2D(8, (3, 3), strides=(1, 1), padding="same", activation="relu")(inputs)
    x = layers.Conv2D(16, (3, 3), strides=(2, 1), padding="same", activation="relu")(x)

    freq_dim = model_config.input_shape[1]
    time_dim = model_config.input_shape[0] // 2

    x = layers.Reshape((time_dim, freq_dim * 16))(x)
    x = layers.Dense(model_config.embed_dim, activation="relu")(x)

    # 2. Positional Encoding
    positions = tf.range(start=0, limit=time_dim, delta=1)
    pos_encoding = layers.Embedding(input_dim=time_dim, output_dim=model_config.embed_dim)(positions)
    x = x + pos_encoding

    # 3. Custom Multi-Head Attention
    key_dim = model_config.embed_dim // model_config.num_heads
    attn_output = custom_multihead_attention(x, num_heads=model_config.num_heads, key_dim=key_dim)
    x = layers.BatchNormalization()(x + attn_output)

    # 4. Feed-forward
    ffn_output = layers.Dense(model_config.ff_dim, activation="relu")(x)
    ffn_output = layers.Dropout(0.1)(ffn_output)
    ffn_output = layers.Dense(model_config.embed_dim)(ffn_output)
    x = layers.BatchNormalization()(x + ffn_output)

    # 5. Global average pooling và Output
    x = layers.GlobalAveragePooling1D()(x)
    x = layers.Dense(model_config.ff_dim, activation="relu")(x)
    x = layers.Dropout(0.2)(x)
    outputs = layers.Dense(model_config.num_classes, activation="softmax")(x)

    return tf.keras.Model(inputs=inputs, outputs=outputs)

In [ ]:
# Preview Model
model = build_kws_transformer_light(model_config)
model.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"])
model.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, 100, 40,   │          0 │ -                 │
│ (InputLayer)        │ 1)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d (Conv2D)     │ (None, 100, 40,   │         80 │ input_layer[0][0] │
│                     │ 8)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_1 (Conv2D)   │ (None, 50, 40,    │      1,168 │ conv2d[0][0]      │
│                     │ 16)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ reshape (Reshape)   │ (None, 50, 640)   │          0 │ conv2d_1[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense (Dense)       │ (None, 50, 64)    │     41,024 │ reshape[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add (Add)           │ (None, 50, 64)    │          0 │ dense[0][0]       │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_1 (Dense)     │ (None, 50, 16)    │      1,040 │ add[0][0]         │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_2 (Dense)     │ (None, 50, 16)    │      1,040 │ add[0][0]         │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_4 (Dense)     │ (None, 50, 16)    │      1,040 │ add[0][0]         │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_5 (Dense)     │ (None, 50, 16)    │      1,040 │ add[0][0]         │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_7 (Dense)     │ (None, 50, 16)    │      1,040 │ add[0][0]         │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_8 (Dense)     │ (None, 50, 16)    │      1,040 │ add[0][0]         │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_10 (Dense)    │ (None, 50, 16)    │      1,040 │ add[0][0]         │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_11 (Dense)    │ (None, 50, 16)    │      1,040 │ add[0][0]         │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ reshape_1 (Reshape) │ (None, 50, 1, 16) │          0 │ dense_1[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ reshape_2 (Reshape) │ (None, 1, 50, 16) │          0 │ dense_2[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ reshape_7 (Reshape) │ (None, 50, 1, 16) │          0 │ dense_4[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ reshape_8 (Reshape) │ (None, 1, 50, 16) │          0 │ dense_5[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ reshape_13          │ (None, 50, 1, 16) │          0 │ dense_7[0][0]     │
│ (Reshape)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ reshape_14          │ (None, 1, 50, 16) │          0 │ dense_8[0][0]     │
│ (Reshape)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ reshape_19          │ (None, 50, 1, 16) │          0 │ dense_10[0][0]    │
│ (Reshape)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ reshape_20          │ (None, 1, 50, 16) │          0 │ dense_11[0][0]  

 Total params: 113,701 (444.14 KB)

 Trainable params: 110,181 (430.39 KB)

 Non-trainable params: 3,520 (13.75 KB)

## 4. Training
Chạy tuning FE config cho model gốc được tạo.\
Sau đó lưu best config cùng model (quantize về INT8) để chạy trên ESP32.

In [ ]:
# Các hàm hỗ trợ cho quá trình train model

class TrainingConfig:
    """Configuration for model training"""
    def __init__(self, epochs=15, batch_size=32, validation_split=0.2,
                 early_stopping_patience=5, learning_rate=0.001):
        self.epochs = epochs
        self.batch_size = batch_size
        self.validation_split = validation_split
        self.early_stopping_patience = early_stopping_patience
        self.learning_rate = learning_rate

    def __repr__(self):
        return (f"TrainingConfig(epochs={self.epochs}, batch_size={self.batch_size}, "
                f"validation_split={self.validation_split}, "
                f"early_stopping_patience={self.early_stopping_patience}, "
                f"learning_rate={self.learning_rate})")


def train_model(X_train, y_train, X_test, y_test, model, train_config, fe_config,
                model_name="model", verbose=1):
    """
    Train model and return comprehensive results.

    Args:
        X_train: Training features (samples, time_steps, num_ceps)
        y_train: Training labels
        X_test: Test features
        y_test: Test labels
        model: Keras model to train
        train_config: TrainingConfig instance
        model_name: Name for tracking results
        verbose: Verbosity level

    Returns:
        Dict with results: {
            'model': trained model,
            'model_name': model name,
            'history': training history,
            'test_loss': test loss,
            'test_accuracy': test accuracy,
            'train_accuracy': final training accuracy,
            'y_pred': predictions on test set,
            'fe_config': current FE config,
            'model_config': current model config,
            'train_config': training config
        }
    """
    import time

    # Add channel dimension
    X_train_reshaped = X_train[..., np.newaxis]
    X_test_reshaped = X_test[..., np.newaxis]

    print(f"\n{'='*60}")
    print(f"Training: {model_name}")
    print(f"{'='*60}")
    print(f"X_train shape: {X_train_reshaped.shape}, y_train shape: {y_train.shape}")
    print(f"X_test shape: {X_test_reshaped.shape}, y_test shape: {y_test.shape}")
    print(f"Training Config: {train_config}")

    # Setup optimizer, learning rate
    optimizer = tf.keras.optimizers.Adam(learning_rate=train_config.learning_rate)
    model.compile(
        optimizer=optimizer,
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"]
    )

    # Callbacks
    callbacks = [
        tf.keras.callbacks.EarlyStopping(
            monitor='val_accuracy',
            patience=train_config.early_stopping_patience,
            restore_best_weights=True,
            verbose=1
        )
    ]

    # Train
    start_time = time.time()
    history = model.fit(
        X_train_reshaped,
        y_train,
        epochs=train_config.epochs,
        batch_size=train_config.batch_size,
        validation_split=train_config.validation_split,
        callbacks=callbacks,
        verbose=verbose
    )
    training_time = time.time() - start_time

    # Evaluate
    train_loss, train_accuracy = model.evaluate(X_train_reshaped, y_train, verbose=0)
    test_loss, test_accuracy = model.evaluate(X_test_reshaped, y_test, verbose=0)

    # Predictions
    y_pred_probs = model.predict(X_test_reshaped, verbose=0)
    y_pred = np.argmax(y_pred_probs, axis=1)

    print(f"\nTraining completed in {training_time:.2f}s")
    print(f"Train Loss: {train_loss:.4f}, Train Accuracy: {train_accuracy:.4f}")
    print(f"Test Loss: {test_loss:.4f}, Test Accuracy: {test_accuracy:.4f}")

    return {
        'model': model,
        'model_name': model_name,
        'history': history,
        'test_loss': test_loss,
        'test_accuracy': test_accuracy,
        'train_loss': train_loss,
        'train_accuracy': train_accuracy,
        'training_time': training_time,
        'y_pred': y_pred,
        'y_pred_probs': y_pred_probs,
        'fe_config': fe_config,
        'model_config': model_config,
        'train_config': train_config
    }

Default Training Config: TrainingConfig(epochs=15, batch_size=32, validation_split=0.2, early_stopping_patience=5, learning_rate=0.001)


In [ ]:
# 8 FE config chạy thử để tìm config tốt nhất
# Config tốt nhất sau quá trình chạy là config 7
fe_configs_to_test = [
    # (nfilt, nfft, target_time_steps, config_name)
    # (32, 256, 80, "FE_Config_1: Small (nfilt=32, nfft=256, steps=80)"),
    # (40, 256, 100, "FE_Config_2: Baseline (nfilt=40, nfft=256, steps=100)"),
    # (40, 512, 100, "FE_Config_3: Medium FFT (nfilt=40, nfft=512, steps=100)"),
    # (48, 512, 100, "FE_Config_4: Larger filters (nfilt=48, nfft=512, steps=100)"),
    # (40, 512, 120, "FE_Config_5: More time (nfilt=40, nfft=512, steps=120)"),
    # (64, 512, 100, "FE_Config_6: Large filters (nfilt=64, nfft=512, steps=100)"),
    (32, 512, 100, "FE_Config_7: Small filters (nfilt=32, nfft=512, steps=100)"),
    # (40, 1024, 100, "FE_Config_8: Large FFT (nfilt=40, nfft=1024, steps=100)"),
]

# Model config (tương tự với config gốc ~ 110k params)
default_model_params = {
    'embed_dim': 64,
    'num_heads': 4,
    'ff_dim': 256,
}

# Training config
default_train_params = {
    'epochs': 20,
    'batch_size': 32,
    'validation_split': 0.2,
    'early_stopping_patience': 5,
    'learning_rate': 0.001,
}

In [ ]:
# Utils để chạy tuning và lưu model về .h

def calculate_normalization_stats(X_data, fe_config):
    """
    Tính global mean, std từ train data bằng extract_mel_spectrogram_raw
    Được gọi trước để tính mean, std cho trích xuất đặc trưng
    """
    print(f"\nCalculating normalization stats from {len(X_data)} samples...")
    all_features = []

    for i, filepath in enumerate(X_data):
        try:
            mel = extract_mel_spectrogram_raw(filepath, fe_config)
            all_features.append(mel)
            # if (i + 1) % 100 == 0:
            #     print(f"  Processed {i + 1}/{len(X_data)} files")
        except Exception as e:
            print(f"  Warning: Error processing {filepath}: {e}")
            continue

    all_features = np.array(all_features)
    global_mean = np.mean(all_features, axis=(0, 1))
    global_std = np.std(all_features, axis=(0, 1))

    print(f"  Global mean shape: {global_mean.shape}, Global std shape: {global_std.shape}")
    return global_mean, global_std


def extract_features_with_norm(X_data, fe_config, global_mean, global_std):
    """
    Trích xuất đặc trưng (có normalize), được gọi sau calculate_normalization_stats(...)
    """
    features = []
    for i, filepath in enumerate(X_data):
        try:
            mel = extract_mel_spectrogram(filepath, fe_config, global_mean, global_std)
            features.append(mel)
            # if (i + 1) % 100 == 0:
            #     print(f"  Processed {i + 1}/{len(X_data)} files")
        except Exception as e:
            print(f"  Warning: Error processing {filepath}: {e}")
            continue

    return np.array(features)


def tune_fe_pipeline(X_train_data, y_train_data, X_test_data, y_test_data,
                     fe_configs, model_config_dict, train_config_dict, verbose=0):
    """
    Thử nghiệm các config FE trong fe_configs để tìm config tốt nhất.

    Returns:
        List kết quả cho các FE configs.
    """
    from sklearn.metrics import accuracy_score

    fe_results = []

    for i, (nfilt, nfft, target_time_steps, config_name) in enumerate(fe_configs):
        print(f"\n{'='*70}")
        print(f"[{i+1}/{len(fe_configs)}] Testing: {config_name}")
        print(f"{'='*70}")

        # Tạo FE config
        fe_cfg = FEConfig(nfilt=nfilt, nfft=nfft, target_time_steps=target_time_steps)
        print(f"FE Config: {fe_cfg}")

        # 1: Tính mean, std từ tập TRAIN
        print("Step 1: Calculating normalization stats from training set...")
        global_mean, global_std = calculate_normalization_stats(X_train_data, fe_cfg)

        # 2: Trích xuất đặc trưng tập train
        print("Step 2: Extracting and normalizing training features...")
        X_train_features = extract_features_with_norm(X_train_data, fe_cfg, global_mean, global_std)
        print(f"  X_train_features shape: {X_train_features.shape}")

        # 3: Trích xuất đặc trưng tập test
        print("Step 3: Extracting and normalizing test features...")
        X_test_features = extract_features_with_norm(X_test_data, fe_cfg, global_mean, global_std)
        print(f"  X_test_features shape: {X_test_features.shape}")

        # Build model
        print("Step 4: Building and training model...")
        model_cfg = ModelConfig(
            input_shape=(fe_cfg.target_time_steps, fe_cfg.nfilt, 1),
            num_classes=len(np.unique(y_train_data)),
            embed_dim=model_config_dict['embed_dim'],
            num_heads=model_config_dict['num_heads'],
            ff_dim=model_config_dict['ff_dim'],
        )
        mod = build_kws_transformer_light(model_cfg)
        mod.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"])

        # Train model
        train_cfg = TrainingConfig(**train_config_dict)
        result = train_model(
            X_train_features, y_train_data,
            X_test_features, y_test_data,
            mod, train_cfg, fe_cfg,
            model_name=config_name,
            verbose=verbose,
        )

        # Lưu kết quả
        result['config_name'] = config_name
        result['test_accuracy_score'] = accuracy_score(y_test_data, result['y_pred'])
        result['global_mean'] = global_mean
        result['global_std'] = global_std

        fe_results.append(result)

        print(f"\nResult: Test Accuracy = {result['test_accuracy']:.4f}, Training Time = {result['training_time']:.2f}s")

    return fe_results


def compare_fe_results(fe_results, metric='test_accuracy'):
    """
    So sánh kết quả các FE configs.
    """
    import pandas as pd

    comparison_data = []
    for result in fe_results:
        comparison_data.append({
            'Config': result['config_name'],
            'Test Acc': result['test_accuracy'],
            'Train Acc': result['train_accuracy'],
            'Test Loss': result['test_loss'],
            'Train Loss': result['train_loss'],
            'Time (s)': result['training_time'],
            'FE Params': f"nfilt={result['fe_config'].nfilt}, nfft={result['fe_config'].nfft}, steps={result['fe_config'].target_time_steps}",
        })

    comparison_df = pd.DataFrame(comparison_data)

    if metric == 'test_accuracy':
        comparison_df = comparison_df.sort_values('Test Acc', ascending=False)
    elif metric == 'test_loss':
        comparison_df = comparison_df.sort_values('Test Loss', ascending=True)

    print(f"\n{'='*120}")
    print(f"FE Configuration Comparison (sorted by {metric})")
    print(f"{'='*120}")
    print(comparison_df.to_string(index=False))
    print(f"{'='*120}\n")

    return comparison_df


def save_results(fe_results, save_path="./keyword_spot/fe_tuning_results.txt"):
    """Lưu kết quả tuning"""
    with open(save_path, 'w') as f:
        for result in fe_results:
            f.write(f"\n{'='*60}\n")
            f.write(f"Config: {result['config_name']}\n")
            f.write(f"{'='*60}\n")
            f.write(f"FE Config: {result['fe_config']}\n")
            f.write(f"Model Config: {result['model_config']}\n")
            f.write(f"Train Config: {result['train_config']}\n")
            f.write(f"Test Accuracy: {result['test_accuracy']:.6f}\n")
            f.write(f"Test Loss: {result['test_loss']:.6f}\n")
            f.write(f"Train Accuracy: {result['train_accuracy']:.6f}\n")
            f.write(f"Train Loss: {result['train_loss']:.6f}\n")
            f.write(f"Training Time: {result['training_time']:.2f}s\n")

    print(f"Results saved to {save_path}")


def export_model_to_h(best_result, model_name="kws_700k", save_dir="./keyword_spot"):
    """Lưu model về dạng .h để chạy được trên ESP32"""
    import os
    import numpy as np
    import tensorflow as tf

    os.makedirs(save_dir, exist_ok=True)

    # Lấy model và các thông số trích xuất đặc trưng từ best_result
    best_model = best_result['model']
    fe_config = best_result['fe_config']
    global_mean = best_result['global_mean']
    global_std = best_result['global_std']

    print(f"\n{'='*70}")
    print(f"EXPORTING MODEL TO .H FILE")
    print(f"{'='*70}")

    print("\nConverting model to TFLite (Full INT8)...")

    # Hàm tạo tập dữ liệu mẫu: Trích xuất trực tiếp từ X_train
    def representative_dataset():
        # Lấy 100 mẫu đầu tiên từ tập X_train (X_train.values chứa danh sách đường dẫn file)
        for i in range(100):
            filepath = X_train.values[i]

            # Trích xuất Mel Spectrogram và chuẩn hóa y như lúc Train
            mel = extract_mel_spectrogram(filepath, fe_config, global_mean, global_std)

            # Reshape thành (1, time_steps, nfilt, 1) để khớp với Input Layer của Model
            sample = np.expand_dims(np.expand_dims(mel, axis=0), axis=-1).astype(np.float32)
            yield [sample]

    # Convert sang Full INT8 (Quantize để giảm kích thước)
    converter = tf.lite.TFLiteConverter.from_keras_model(best_model)
    converter.optimizations = [tf.lite.Optimize.DEFAULT]
    converter.representative_dataset = representative_dataset

    # BẮT BUỘC toàn bộ Tensor trung gian phải được ép về INT8
    converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
    converter.inference_input_type = tf.int8
    converter.inference_output_type = tf.int8

    tflite_model = converter.convert()

    model_size_kb = len(tflite_model) / 1024
    print(f"TFLite model size: {model_size_kb:.2f} KB")

    # Chuyển sang file .h
    print(f"\nExporting to .h header file...")
    h_file_path = os.path.join(save_dir, f"{model_name}.h")

    hex_lines = [f"0x{byte:02x}" for byte in tflite_model]

    with open(h_file_path, "w") as f:
        f.write(f"#ifndef {model_name.upper()}_H\n")
        f.write(f"#define {model_name.upper()}_H\n\n")
        f.write(f"const unsigned char {model_name}[] __attribute__((aligned(16))) = {{\n")

        for i, hex_val in enumerate(hex_lines):
            if i % 12 == 0:
                f.write("  ")
            f.write(hex_val)
            if i < len(hex_lines) - 1:
                f.write(", ")
            if (i + 1) % 12 == 0 and i < len(hex_lines) - 1:
                f.write("\n")

        f.write("\n};\n\n")
        f.write(f"const int {model_name}_len = {len(tflite_model)};\n")
        f.write(f"#endif\n")

    print(f"Saved to: {h_file_path}")
    print(f"{'='*70}\n")

In [ ]:
# RUN FE TUNING (enable TUNE_FE flag to execute)
if TUNE_FE:
    print("\n" + "="*70)
    print("STARTING FE CONFIGURATION TUNING")
    print("="*70)

    fe_tuning_results = tune_fe_pipeline(
        X_train_data=X_train.values,
        y_train_data=y_train.values,
        X_test_data=X_test.values,
        y_test_data=y_test.values,
        fe_configs=fe_configs_to_test,
        model_config_dict=default_model_params,
        train_config_dict=default_train_params,
        verbose=0,
    )

    comparison_results = compare_fe_results(fe_tuning_results, metric='test_accuracy')
    save_results(fe_tuning_results, save_path="./keyword_spot/kws_150k_3.txt")

    best_result = max(fe_tuning_results, key=lambda x: x['test_accuracy'])
    print(f"\n{'='*70}")
    print(f"BEST FE CONFIG: {best_result['config_name']}")
    print(f"  Test Accuracy: {best_result['test_accuracy']:.4f}")
    print(f"  Training Time: {best_result['training_time']:.2f}s")
    print(f"  FE Config: {best_result['fe_config']}")
    print(f"{'='*70}")

    # Save best model to .h file
    export_model_to_h(best_result, model_name="kws_150k_3", save_dir="./keyword_spot")
else:
    print("TUNE_FE=False, skipped FE tuning")


STARTING FE CONFIGURATION TUNING

[1/1] Testing: FE_Config_7: Small filters (nfilt=32, nfft=512, steps=100)
FE Config: FEConfig(nfilt=32, nfft=512, target_time_steps=100)
Step 1: Calculating normalization stats from training set...

Calculating normalization stats from 9441 samples...
  Global mean shape: (32,), Global std shape: (32,)
Step 2: Extracting and normalizing training features...
  X_train_features shape: (9441, 100, 32)
Step 3: Extracting and normalizing test features...
  X_test_features shape: (2361, 100, 32)
Step 4: Building and training model...

Training: FE_Config_7: Small filters (nfilt=32, nfft=512, steps=100)
X_train shape: (9441, 100, 32, 1), y_train shape: (9441,)
X_test shape: (2361, 100, 32, 1), y_test shape: (2361,)
Training Config: TrainingConfig(epochs=20, batch_size=32, validation_split=0.2, early_stopping_patience=5, learning_rate=0.001)
Restoring model weights from the end of the best epoch: 19.

Training completed in 56.21s
Train Loss: 0.0989, Train Acc

/usr/local/lib/python3.12/dist-packages/tensorflow/lite/python/convert.py:863: UserWarning: Statistics for quantized inputs were expected, but not specified; continuing anyway.
  warnings.warn(


TFLite model size: 163.70 KB

Exporting to .h header file...
Saved to: ./keyword_spot/kws_150k_3.h



## ???

In [ ]:
import numpy as np

# Các thông số của best model
nfilt = 32
nfft = 512

# 1. Export Hamming Window y hệt lúc train
window = np.hamming(400)

# 2. Xây dựng lại CHÍNH XÁC mảng fbank đã dùng lúc Train (KHÔNG DÙNG LIBROSA)
low_freq_mel = 0
high_freq_mel = (2595 * np.log10(1 + (sample_rate / 2) / 700))
mel_points = np.linspace(low_freq_mel, high_freq_mel, nfilt + 2)
hz_points = (700 * (10**(mel_points / 2595) - 1))
bin = np.floor((nfft + 1) * hz_points / sample_rate)

fbank = np.zeros((nfilt, int(np.floor(nfft / 2 + 1))))
for m in range(1, nfilt + 1):
    f_m_minus = int(bin[m - 1])
    f_m = int(bin[m])
    f_m_plus = int(bin[m + 1])
    for k in range(f_m_minus, f_m):
        fbank[m - 1, k] = (k - bin[m - 1]) / (bin[m] - bin[m - 1])
    for k in range(f_m, f_m_plus):
        fbank[m - 1, k] = (bin[m + 1] - k) / (bin[m + 1] - bin[m])

mel_basis = fbank # Thay thế librosa bằng fbank tự làm

# global_mean và global_std lấy từ best_result
global_mean = best_result['global_mean']
global_std = best_result['global_std']

# 3. Ghi ra file C++ Header
with open("feature_extraction_data.h", "w") as f:
    f.write("#ifndef FEATURE_EXTRACTION_DATA_H\n")
    f.write("#define FEATURE_EXTRACTION_DATA_H\n\n")

    # Write Hamming Window
    f.write(f"const float hamming_window[{len(window)}] = {{\n    ")
    f.write(", ".join([f"{x:.6f}f" for x in window]))
    f.write("\n};\n\n")

    # Write Mel Filters
    f.write(f"const float mel_filters[{nfilt}][{nfft // 2 + 1}] = {{\n")
    for i in range(nfilt):
        f.write("    {")
        f.write(", ".join([f"{x:.6f}f" for x in mel_basis[i]]))
        f.write("}")
        if i < nfilt - 1:
            f.write(",\n")
        else:
            f.write("\n")
    f.write("};\n\n")

    # Write Global Mean
    f.write(f"const float global_mean[{nfilt}] = {{\n    ")
    f.write(", ".join([f"{x:.6f}f" for x in global_mean]))
    f.write("\n};\n\n")

    # Write Global Std
    f.write(f"const float global_std[{nfilt}] = {{\n    ")
    f.write(", ".join([f"{x:.6f}f" for x in global_std]))
    f.write("\n};\n\n")

    f.write("#endif // FEATURE_EXTRACTION_DATA_H\n")

print("File feature_extraction_data.h đã được tạo thành công!")


In [ ]:
import numpy as np
import scipy.io.wavfile as wav
import pandas as pd
import os

# 1. Xác định số lượng file cần lấy cho mỗi nhãn
target_counts = {
    'down': 3,
    'left': 3,
    'right': 3,
    'up': 3,
    'other': 10
}

# 2. Lọc file từ tập X_test (tập test AI chưa từng thấy khi train)
test_df = pd.DataFrame({
    'filepath': X_test.values,
    'label_idx': y_test.values
})
test_df['label_str'] = le.inverse_transform(test_df['label_idx'])

selected_files = []
for label, count in target_counts.items():
    label_files = test_df[test_df['label_str'] == label]['filepath'].values
    np.random.shuffle(label_files) # Đảo ngẫu nhiên để lấy file bất kỳ
    selected = label_files[:count]
    for f in selected:
        selected_files.append((label, f))

# 3. Xuất ra file C++ Header
h_file_path = "test_audio_data.h"
with open(h_file_path, "w") as f:
    f.write("#ifndef TEST_AUDIO_DATA_H\n")
    f.write("#define TEST_AUDIO_DATA_H\n\n")
    f.write("#include <stdint.h>\n\n")

    f.write("typedef struct {\n")
    f.write("    const char* label;\n")
    f.write("    const int16_t data[16000];\n")
    f.write("} TestAudioSample;\n\n")

    f.write(f"const int NUM_TEST_SAMPLES = {len(selected_files)};\n\n")
    f.write("const TestAudioSample test_samples[] = {\n")

    for i, (label, filepath) in enumerate(selected_files):
        sr, signal = wav.read(filepath)

        # Cắt hoặc chèn thêm số 0 cho chuẩn 16000 mẫu (1 giây)
        if len(signal) < 16000:
            signal = np.pad(signal, (0, 16000 - len(signal)), 'constant')
        elif len(signal) > 16000:
            signal = signal[:16000]

        f.write("    {\n")
        f.write(f'        "{label}",\n')
        f.write("        {")

        for j in range(16000):
            if j % 20 == 0:
                f.write("\n            ")
            f.write(f"{int(signal[j])}, ")
        f.write("\n        }\n")
        f.write("    }")

        if i < len(selected_files) - 1:
            f.write(",")
        f.write("\n")

    f.write("};\n\n")
    f.write("#endif // TEST_AUDIO_DATA_H\n")

print(f"Thành công! Đã xuất {len(selected_files)} file wav ra {h_file_path}")


Thành công! Đã xuất 22 file wav ra test_audio_data.h
